In [ ]:
%pip install xgboost
dbutils.library.restartPython()

# ATP Model Serving — Register & Deploy

Wraps the **three XGBoost models** from `atp_model.ipynb` (two per-side serve regressors + one direct match-outcome classifier) into a single MLflow `pyfunc` model and deploys it as a live Databricks Model Serving endpoint.

**Architecture (mirrors `predict_match()` in `atp_model.ipynb`):**
- Stage 1: `serve_model_p1` + `serve_model_p2` → per-side serve-win probabilities `p`, `q`
- Stage 2: Markov chain over scoring tree → raw match prob → Platt-calibrated → `p_markov_cal`
- Stage 2b: `direct_model` → `p_direct`
- Headline output: 50/50 ensemble `0.5 * p_markov_cal + 0.5 * p_direct`

**Inputs to the live endpoint (simple API):** player names + match context. Player feature snapshots are bundled with the model so the endpoint doesn't need access to the gold table at runtime.

**Outputs:**
- Registered UC model: `workspace.default.atp_match_predictor` (alias: `production`)
- Live endpoint: `atp-predictor`
- `/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/serving_url.txt` for downstream use

## Section 1 — Imports & paths

In [ ]:
import os
import time
import json
import inspect
import requests

import numpy as np
import pandas as pd

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.utils.databricks_utils import get_databricks_host_creds

# === Free Edition / Serverless fix =========================================
# Spark Connect on Free Edition does not expose `spark.mlflow.modelRegistryUri`,
# so MlflowClient() with default args throws CONFIG_NOT_AVAILABLE.
mlflow.set_tracking_uri('databricks')
mlflow.set_registry_uri('databricks-uc')
# ===========================================================================

EXPERIMENT_PATH       = '/Users/f.chiesa28@ncf.edu/Octenpus/tennis_atp_prediction'
REGISTERED_MODEL_NAME = 'workspace.default.atp_match_predictor'
PROD_ALIAS            = 'production'
ENDPOINT_NAME         = 'atp-predictor'
SERVING_URL_OUT       = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/serving_url.txt'
GOLD_PATH             = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets/atp_gold.parquet'
MODELS_DIR            = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/models'  # .ubj files saved by atp_model.ipynb

# Artifact paths inside the source run — match how `atp_model.ipynb` logs them
ARTIFACT_SERVE_P1 = 'serve_model_p1'
ARTIFACT_SERVE_P2 = 'serve_model_p2'
ARTIFACT_DIRECT   = 'direct_model'

mlflow.set_experiment(EXPERIMENT_PATH)
client = MlflowClient(tracking_uri='databricks', registry_uri='databricks-uc')
print('MLflow ready.')
print(f'  tracking : {mlflow.get_tracking_uri()}')
print(f'  registry : {mlflow.get_registry_uri()}')

## Section 2 — Markov chain helper (`/tmp/markov_helper.py`)

Self-contained `.py` so MLflow's `code_paths` mechanism can bundle it. Game level: closed form. Tiebreak: iterative bottom-up DP with 50-pt cap. Set / match: memoized DP.

In [ ]:
markov_helper_source = """\
\"\"\"Markov chain over the tennis scoring tree.\"\"\"
from functools import lru_cache


def _game_prob(s):
    if s <= 0: return 0.0
    if s >= 1: return 1.0
    sq = 1.0 - s
    denom = s * s + sq * sq
    deuce = (s * s) / denom if denom > 0 else 0.5
    return s**4 + 4 * s**4 * sq + 10 * s**4 * sq**2 + 20 * s**3 * sq**3 * deuce


def _tiebreak_prob(p, q, max_points=50):
    cache = {}
    for total in range(max_points, -1, -1):
        for s1 in range(total + 1):
            s2 = total - s1
            if s1 >= 7 and s1 - s2 >= 2:
                cache[(s1, s2)] = 1.0; continue
            if s2 >= 7 and s2 - s1 >= 2:
                cache[(s1, s2)] = 0.0; continue
            if total == max_points:
                cache[(s1, s2)] = 0.5; continue
            idx = total
            if idx == 0:
                win_p1_pt = p
            else:
                pair = (idx - 1) // 2
                win_p1_pt = (1.0 - q) if pair % 2 == 0 else p
            cache[(s1, s2)] = (win_p1_pt * cache[(s1+1, s2)]
                               + (1.0 - win_p1_pt) * cache[(s1, s2+1)])
    return cache[(0, 0)]


def markov_win_prob(p, q, best_of_5=False):
    p = max(min(float(p), 0.999), 0.001)
    q = max(min(float(q), 0.999), 0.001)
    g_p = _game_prob(p)
    g_q_loss = 1.0 - _game_prob(q)
    tb_prob = _tiebreak_prob(p, q)

    @lru_cache(maxsize=None)
    def set_dp(g1, g2, p1_serves):
        if g1 == 6 and g2 <= 4: return 1.0
        if g2 == 6 and g1 <= 4: return 0.0
        if g1 == 7 and g2 in (5, 6): return 1.0
        if g2 == 7 and g1 in (5, 6): return 0.0
        if g1 == 6 and g2 == 6: return tb_prob
        win_game = g_p if p1_serves else g_q_loss
        return (win_game * set_dp(g1+1, g2, not p1_serves)
                + (1.0 - win_game) * set_dp(g1, g2+1, not p1_serves))

    set_p = set_dp(0, 0, True)
    target = 3 if best_of_5 else 2

    @lru_cache(maxsize=None)
    def match_dp(s1, s2):
        if s1 == target: return 1.0
        if s2 == target: return 0.0
        return set_p * match_dp(s1+1, s2) + (1.0 - set_p) * match_dp(s1, s2+1)

    return match_dp(0, 0)
"""

with open('/tmp/markov_helper.py', 'w') as f:
    f.write(markov_helper_source)

# Sanity
import importlib, sys
sys.path.insert(0, '/tmp')
sys.modules.pop('markov_helper', None)
import markov_helper
print(f'(0.7,0.7) BO3 = {markov_helper.markov_win_prob(0.7, 0.7, False):.4f}  (expect 0.5)')
print(f'(0.75,0.65) BO5 = {markov_helper.markov_win_prob(0.75, 0.65, True):.4f}')

## Section 3 — Pyfunc wrapper

`ATPMatchPredictor` matches `predict_match()` in `atp_model.ipynb` exactly:

- **Loads** three XGBoost models + a player-snapshot dict + Platt params from `context.artifacts`
- **Input columns** (per row): `p1_name, p2_name, surface, is_grand_slam, is_best_of_5`
- **Internal flow:** look up snapshots → build serve / direct feature vectors → run all three models → combine via Markov + Platt → average with direct
- **Output columns:** `p1_name, p2_name, p1_win_prob, p2_win_prob, predicted_winner, p_markov_cal, p_direct, p1_serve_pred, p2_serve_pred, model_version`

Feature lists below mirror exactly `FEATURES_SERVE_P1`, `FEATURES_SERVE_P2`, `FEATURES_DIRECT` and `PLAYER_FEATS_BASE` in `atp_model.ipynb`.

In [ ]:
import mlflow.pyfunc

HAND_MAP    = {'R': 1.0, 'L': 0.0, 'U': 0.5}
SURFACE_MAP = {'Hard': 0, 'Clay': 1, 'Grass': 2, 'Carpet': 3}
SERVE_CLIP  = (0.40, 0.75)

PLAYER_FEATS_BASE = [
    'elo', 'win_rate_last_20', 'win_rate_last_5', 'win_rate_surface_12m',
    'serve_pct_surface', 'first_won_pct_roll', 'second_won_pct_roll',
    'ace_rate', 'matches_prev_tourneys_7d', 'sets_prev_tourneys_7d',
    'days_rest', 'h2h_winrate', 'age', 'hand_enc', 'rank',
]

MODEL_VERSION_TAG = 'ensemble-markov-direct-v1'


class ATPMatchPredictor(mlflow.pyfunc.PythonModel):
    """Three-model ATP match predictor matching atp_model.ipynb predict_match().

    Loads XGBoost models from .ubj files (XGBoost's native binary format) bundled
    as artifacts. Bypasses MLflow's model registry/storage entirely — the .ubj
    files are saved manually by atp_model.ipynb to a known Workspace path.
    """

    P1_SERVE_FEATURES = [
        'p1_elo', 'p1_win_rate_last_20', 'p1_win_rate_last_5',
        'p1_win_rate_surface_12m', 'p1_serve_pct_surface',
        'p1_first_won_pct_roll', 'p1_second_won_pct_roll', 'p1_ace_rate',
        'p1_matches_prev_tourneys_7d', 'p1_sets_prev_tourneys_7d',
        'p1_days_rest', 'p1_h2h_winrate', 'p1_age', 'p1_hand_enc',
        'p2_elo', 'p2_win_rate_last_20', 'p2_win_rate_surface_12m',
        'p2_first_won_pct_roll', 'p2_second_won_pct_roll',
        'p2_ace_rate', 'p2_age', 'p2_hand_enc',
        'is_grand_slam', 'is_best_of_5', 'surface_encoded',
        'elo_diff', 'winrate_diff',
    ]
    P2_SERVE_FEATURES = [
        'p2_elo', 'p2_win_rate_last_20', 'p2_win_rate_last_5',
        'p2_win_rate_surface_12m', 'p2_serve_pct_surface',
        'p2_first_won_pct_roll', 'p2_second_won_pct_roll', 'p2_ace_rate',
        'p2_matches_prev_tourneys_7d', 'p2_sets_prev_tourneys_7d',
        'p2_days_rest', 'p2_h2h_winrate', 'p2_age', 'p2_hand_enc',
        'p1_elo', 'p1_win_rate_last_20', 'p1_win_rate_surface_12m',
        'p1_first_won_pct_roll', 'p1_second_won_pct_roll',
        'p1_ace_rate', 'p1_age', 'p1_hand_enc',
        'is_grand_slam', 'is_best_of_5', 'surface_encoded',
        'elo_diff', 'winrate_diff',
    ]
    DIRECT_FEATURES = [
        'elo_diff', 'rank_diff', 'age_diff', 'winrate_diff', 'serve_diff',
        'fatigue_diff', 'p1_h2h_winrate',
        'is_grand_slam', 'is_best_of_5', 'surface_encoded',
    ]
    PLAYER_FEATS = [
        'elo', 'win_rate_last_20', 'win_rate_last_5', 'win_rate_surface_12m',
        'serve_pct_surface', 'first_won_pct_roll', 'second_won_pct_roll',
        'ace_rate', 'matches_prev_tourneys_7d', 'sets_prev_tourneys_7d',
        'days_rest', 'h2h_winrate', 'age', 'hand_enc', 'rank',
    ]
    SURFACE_TO_ENC = {'Hard': 0, 'Clay': 1, 'Grass': 2, 'Carpet': 3}
    SERVE_CLIP_LO = 0.40
    SERVE_CLIP_HI = 0.75

    def load_context(self, context):
        import xgboost as xgb

        # Load directly from persistent Workspace paths — bypasses MLflow's S3
        # artifact upload, which fails on Free Edition / Serverless.
        BASE = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/models'

        self.xgb_serve_p1 = xgb.XGBRegressor()
        self.xgb_serve_p1.load_model(f'{BASE}/serve_model_p1.ubj')

        self.xgb_serve_p2 = xgb.XGBRegressor()
        self.xgb_serve_p2.load_model(f'{BASE}/serve_model_p2.ubj')

        self.xgb_direct = xgb.XGBClassifier()
        self.xgb_direct.load_model(f'{BASE}/direct_model.ubj')

        with open(f'{BASE}/player_snapshots.json', 'r') as f:
            self.snapshots = json.load(f)

        with open(f'{BASE}/platt_params.json', 'r') as f:
            pp = json.load(f)
        self.platt_coef      = float(pp['coef'])
        self.platt_intercept = float(pp['intercept'])

        from markov_helper import markov_win_prob
        self._markov = markov_win_prob
        print(f'✓ ATPMatchPredictor loaded from {BASE}')

    def _platt_apply(self, raw):
        raw = np.clip(raw, 1e-6, 1 - 1e-6)
        logit = np.log(raw / (1 - raw))
        return 1.0 / (1.0 + np.exp(-(self.platt_coef * logit + self.platt_intercept)))

    def _build_row(self, p1_name, p2_name, surface, is_grand_slam, is_best_of_5):
        p1s = self.snapshots.get(p1_name)
        p2s = self.snapshots.get(p2_name)
        if p1s is None:
            raise ValueError(f'player not in snapshots: {p1_name!r}')
        if p2s is None:
            raise ValueError(f'player not in snapshots: {p2_name!r}')
        row = {
            'is_grand_slam':   int(bool(is_grand_slam)),
            'is_best_of_5':    int(bool(is_best_of_5)),
            'surface_encoded': self.SURFACE_TO_ENC.get(surface, 0),
        }
        for k, v in p1s.items():
            row[f'p1_{k}'] = float(v) if v is not None else np.nan
        for k, v in p2s.items():
            row[f'p2_{k}'] = float(v) if v is not None else np.nan
        row['elo_diff']     = row['p1_elo']               - row['p2_elo']
        row['rank_diff']    = row['p1_rank']              - row['p2_rank']
        row['age_diff']     = row['p1_age']               - row['p2_age']
        row['winrate_diff'] = row['p1_win_rate_last_20']  - row['p2_win_rate_last_20']
        row['serve_diff']   = row['p1_serve_pct_surface'] - row['p2_serve_pct_surface']
        row['fatigue_diff'] = (row['p1_matches_prev_tourneys_7d']
                              - row['p2_matches_prev_tourneys_7d'])
        return row

    def predict(self, context, model_input):
        df = pd.DataFrame(model_input).copy()
        out_rows = []
        for _, r in df.iterrows():
            try:
                row = self._build_row(
                    r['p1_name'], r['p2_name'], r.get('surface', 'Hard'),
                    r.get('is_grand_slam', False), r.get('is_best_of_5', False),
                )
            except ValueError as e:
                out_rows.append({
                    'p1_name': r['p1_name'], 'p2_name': r['p2_name'],
                    'p1_win_prob': np.nan, 'p2_win_prob': np.nan,
                    'predicted_winner': None,
                    'p_markov_cal': np.nan, 'p_direct': np.nan,
                    'p1_serve_pred': np.nan, 'p2_serve_pred': np.nan,
                    'model_version': MODEL_VERSION_TAG, 'error': str(e),
                })
                continue
            one = pd.DataFrame([row])
            X1   = one[self.P1_SERVE_FEATURES].astype(float).values
            X2   = one[self.P2_SERVE_FEATURES].astype(float).values
            Xdir = one[self.DIRECT_FEATURES].astype(float).values

            p_serve_p1 = float(np.clip(self.xgb_serve_p1.predict(X1)[0],
                                        self.SERVE_CLIP_LO, self.SERVE_CLIP_HI))
            p_serve_p2 = float(np.clip(self.xgb_serve_p2.predict(X2)[0],
                                        self.SERVE_CLIP_LO, self.SERVE_CLIP_HI))

            raw = self._markov(p_serve_p1, p_serve_p2,
                               best_of_5=bool(r.get('is_best_of_5', False)))
            p_markov_cal = float(self._platt_apply(np.array([raw]))[0])
            p_direct     = float(self.xgb_direct.predict_proba(Xdir)[0, 1])
            p_ensemble   = 0.5 * p_markov_cal + 0.5 * p_direct
            winner       = r['p1_name'] if p_ensemble >= 0.5 else r['p2_name']

            out_rows.append({
                'p1_name': r['p1_name'], 'p2_name': r['p2_name'],
                'p1_win_prob': p_ensemble, 'p2_win_prob': 1 - p_ensemble,
                'predicted_winner': winner,
                'p_markov_cal': p_markov_cal, 'p_direct': p_direct,
                'p1_serve_pred': p_serve_p1, 'p2_serve_pred': p_serve_p2,
                'model_version': MODEL_VERSION_TAG,
            })
        return pd.DataFrame(out_rows)


print('ATPMatchPredictor defined.')

## Section 4 — Build snapshots, fetch Platt, log + register

Steps:
1. Read latest player snapshots from the gold parquet (one row per player, most recent match)
2. Fetch `platt_coef` and `platt_intercept` from the source run's logged metrics
3. Save snapshots and Platt params to disk
4. Log the pyfunc with all artifacts → register to UC → alias `production`

## Section 4 (pre) — Clean up broken model versions

Failed `log_model` attempts can leave versions stuck in `PENDING_REGISTRATION` or other non-READY states. Delete those before re-registering. UC versions always report `current_stage == 'None'`, so we filter on `status != 'READY'` only — deleting on stage would nuke the working production version too.

In [ ]:
# Reuse the `client` initialized in Section 1 — MlflowClient() with default
# args raises CONFIG_NOT_AVAILABLE on Free Edition (see cell 2).
broken = [v for v in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
          if v.status != 'READY']
for v in broken:
    client.delete_model_version(REGISTERED_MODEL_NAME, v.version)
    print(f'Deleted broken v{v.version} (status={v.status})')
print(f'Cleanup done. Removed {len(broken)} broken version(s).')

In [ ]:
import shutil

# 4a) Verify the .ubj files saved by atp_model.ipynb exist
required_files = {
    'xgb_serve_p1':     f'{MODELS_DIR}/serve_model_p1.ubj',
    'xgb_serve_p2':     f'{MODELS_DIR}/serve_model_p2.ubj',
    'xgb_direct':       f'{MODELS_DIR}/direct_model.ubj',
    'platt_params':     f'{MODELS_DIR}/platt_params.json',
}
print(f'Looking in {MODELS_DIR}:')
for k, p in required_files.items():
    exists = os.path.exists(p)
    size = os.path.getsize(p) if exists else 0
    flag = '✓' if exists else '✗'
    print(f'  {flag} {k}: {p}  ({size:,} bytes)')
assert all(os.path.exists(p) for p in required_files.values()), (
    f'Some required files missing in {MODELS_DIR}. '
    'Re-run the save cell in atp_model.ipynb that writes the .ubj files + platt_params.json.'
)

# 4b) Sanity-check load each .ubj
import xgboost as xgb
_m = xgb.XGBRegressor(); _m.load_model(required_files['xgb_serve_p1'])
print(f'  ✓ serve_p1 loadable, num_features={_m.get_booster().num_features()}')
_m = xgb.XGBClassifier(); _m.load_model(required_files['xgb_direct'])
print(f'  ✓ direct loadable, num_features={_m.get_booster().num_features()}')

# 4c) Load Platt params
with open(required_files['platt_params']) as f:
    pp = json.load(f)
PLATT_COEF      = float(pp['coef'])
PLATT_INTERCEPT = float(pp['intercept'])
print(f'  Platt: coef={PLATT_COEF:+.4f}  intercept={PLATT_INTERCEPT:+.4f}')

# 4d) Build per-player latest snapshots from gold parquet
print('\nReading gold parquet...')
gold = pd.read_parquet(GOLD_PATH, engine='pyarrow')
gold['match_date'] = pd.to_datetime(gold['match_date'])
print(f'  rows={len(gold):,}  date range: {gold.match_date.min().date()} → {gold.match_date.max().date()}')

if 'p1_hand_enc' not in gold.columns:
    for side in ('p1', 'p2'):
        gold[f'{side}_hand_enc'] = gold[f'{side}_hand'].map(HAND_MAP).fillna(0.5)

print('Building per-player latest snapshots...')
snapshots = {}
for side in ('p1', 'p2'):
    name_col = f'{side}_name'
    if name_col not in gold.columns:
        continue
    feat_cols = [f'{side}_{f}' for f in PLAYER_FEATS_BASE if f'{side}_{f}' in gold.columns]
    sub = gold[[name_col, 'match_date'] + feat_cols].sort_values('match_date').dropna(subset=[name_col])
    latest = sub.groupby(name_col).tail(1)
    for _, r in latest.iterrows():
        name = r[name_col]
        prev = snapshots.get(name)
        if prev is None or pd.to_datetime(prev.get('_match_date', '1900-01-01')) < r['match_date']:
            snap = {f: (None if pd.isna(r.get(f'{side}_{f}', np.nan)) else float(r[f'{side}_{f}']))
                    for f in PLAYER_FEATS_BASE}
            snap['_match_date'] = str(r['match_date'].date())
            snapshots[name] = snap
for name in list(snapshots.keys()):
    snapshots[name].pop('_match_date', None)
print(f'  built {len(snapshots):,} player snapshots')

# 4e) Write snapshots to MODELS_DIR (Workspace) so the served pyfunc can
# read them at serve time without going through MLflow's S3 artifact store.
SNAP_PATH = f'{MODELS_DIR}/player_snapshots.json'
with open(SNAP_PATH, 'w') as f:
    json.dump(snapshots, f)
print(f'  snapshots → {SNAP_PATH}  ({os.path.getsize(SNAP_PATH):,} bytes)')

# 4f) Log + register pyfunc. No `artifacts=` dict: the .ubj/.json files live
# at fixed Workspace paths and are loaded directly by load_context(), avoiding
# MLflow's S3 artifact upload (which 403s on Free Edition / Serverless).
_log_params = inspect.signature(mlflow.pyfunc.log_model).parameters
CODE_KW = 'code_paths' if 'code_paths' in _log_params else 'code_path'
NAME_KW = 'artifact_path' if 'artifact_path' in _log_params else 'name'

sample_input = pd.DataFrame([{
    'p1_name': next(iter(snapshots.keys())),
    'p2_name': list(snapshots.keys())[1] if len(snapshots) > 1 else next(iter(snapshots.keys())),
    'surface': 'Hard',
    'is_grand_slam': False,
    'is_best_of_5':  False,
}])

# Sample output mirrors ATPMatchPredictor.predict() return columns so the
# Unity Catalog signature has both input + output schemas.
from mlflow.models.signature import infer_signature
sample_output = pd.DataFrame([{
    'p1_name':          sample_input.loc[0, 'p1_name'],
    'p2_name':          sample_input.loc[0, 'p2_name'],
    'p1_win_prob':      0.6,
    'p2_win_prob':      0.4,
    'predicted_winner': sample_input.loc[0, 'p1_name'],
    'p_markov_cal':     0.55,
    'p_direct':         0.65,
    'p1_serve_pred':    0.65,
    'p2_serve_pred':    0.60,
    'model_version':    MODEL_VERSION_TAG,
}])
signature = infer_signature(sample_input, sample_output)

log_kwargs = {
    NAME_KW:                 'atp_match_predictor',
    'python_model':          ATPMatchPredictor(),
    'input_example':         sample_input,
    'signature':             signature,
    'registered_model_name': REGISTERED_MODEL_NAME,
    'pip_requirements':      [
        f'xgboost=={xgb.__version__}',
        'pandas',
        'numpy',
        'scikit-learn',
    ],
    CODE_KW:                 ['/tmp/markov_helper.py'],
}

with mlflow.start_run(run_name='register-pyfunc-wrapper'):
    logged = mlflow.pyfunc.log_model(**log_kwargs)
    print(f'\n  → logged + registered as {REGISTERED_MODEL_NAME}')

# 4g) Alias latest version → production
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)
client.update_model_version(
    name=REGISTERED_MODEL_NAME, version=latest_version,
    description='Three-model ATP match predictor (Markov-cal + Direct ensemble).',
)
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME, alias=PROD_ALIAS, version=str(latest_version),
)
print(f'✓ {REGISTERED_MODEL_NAME} v{latest_version} aliased as @{PROD_ALIAS}')

## Section 5 — Create or update the Databricks Model Serving endpoint

Hits `/api/2.0/serving-endpoints` directly with `get_databricks_host_creds()` for auth — no hardcoded tokens. Polls until READY.

> **Free Edition note:** if endpoint creation returns 403 / quota error, Model Serving is not available on your tier. Pivot to FastAPI + a free hosting tier (Hugging Face Spaces, Vercel, Render) — the pyfunc model URI is reusable from any compute that can hit the MLflow registry.

In [ ]:
creds = get_databricks_host_creds()
HOST  = creds.host.rstrip('/')
TOKEN = creds.token
HEADERS = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

prod_mv = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, PROD_ALIAS)
prod_version = prod_mv.version
print(f'Serving {REGISTERED_MODEL_NAME} v{prod_version} (alias @{PROD_ALIAS})')

served_entities = [{
    'name': 'atp_predictor_v1',
    'entity_name': REGISTERED_MODEL_NAME,
    'entity_version': prod_version,
    'workload_size': 'Small',
    'scale_to_zero_enabled': True,
}]

get_resp = requests.get(f'{HOST}/api/2.0/serving-endpoints/{ENDPOINT_NAME}', headers=HEADERS)

if get_resp.status_code == 200:
    print(f'Endpoint {ENDPOINT_NAME!r} exists — updating config')
    upd = requests.put(
        f'{HOST}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config',
        headers=HEADERS, json={'served_entities': served_entities},
    )
    upd.raise_for_status()
elif get_resp.status_code == 404:
    print(f'Creating endpoint {ENDPOINT_NAME!r}')
    cre = requests.post(
        f'{HOST}/api/2.0/serving-endpoints',
        headers=HEADERS,
        json={'name': ENDPOINT_NAME, 'config': {'served_entities': served_entities}},
    )
    cre.raise_for_status()
else:
    get_resp.raise_for_status()

print('Waiting for endpoint to become READY...')
for i in range(60):
    s = requests.get(f'{HOST}/api/2.0/serving-endpoints/{ENDPOINT_NAME}', headers=HEADERS).json()
    state = s.get('state', {})
    ready = state.get('ready'); update = state.get('config_update')
    print(f'  [{i*10:>3}s] ready={ready}  config_update={update}')
    if ready == 'READY' and update in (None, 'NOT_UPDATING'):
        break
    time.sleep(10)
else:
    raise RuntimeError('Endpoint did not become READY within 10 min')

SERVING_URL = f'{HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations'
print(f'\n✓ Serving URL: {SERVING_URL}')

## Section 6 — Smoke test against the live endpoint

POSTs three real matchups (Alcaraz–Sinner, Djokovic–Nadal, etc.) and prints the full JSON response. Asserts HTTP 200.

In [ ]:
test_matches = [
    {'p1_name': 'Carlos Alcaraz', 'p2_name': 'Jannik Sinner',
     'surface': 'Clay',  'is_grand_slam': True,  'is_best_of_5': True},
    {'p1_name': 'Carlos Alcaraz', 'p2_name': 'Jannik Sinner',
     'surface': 'Hard',  'is_grand_slam': False, 'is_best_of_5': False},
    {'p1_name': 'Novak Djokovic', 'p2_name': 'Rafael Nadal',
     'surface': 'Clay',  'is_grand_slam': True,  'is_best_of_5': True},
]

print('POST payload:')
print(json.dumps({'dataframe_records': test_matches}, indent=2))

resp = requests.post(SERVING_URL, headers=HEADERS,
                     json={'dataframe_records': test_matches}, timeout=60)
print(f'\nStatus: {resp.status_code}')
try:
    print('Response:'); print(json.dumps(resp.json(), indent=2))
except Exception:
    print(resp.text)

assert resp.status_code == 200, f'Smoke test failed: {resp.status_code} — {resp.text}'
print('\n✓ Smoke test passed')

## Section 7 — Export the serving URL

So the frontend, `test_project.py`, and any orchestration scripts can read it without hardcoding.

In [ ]:
os.makedirs(os.path.dirname(SERVING_URL_OUT), exist_ok=True)
with open(SERVING_URL_OUT, 'w') as f:
    f.write(SERVING_URL + '\n')
print(f'Wrote serving URL to: {SERVING_URL_OUT}')
print(f'  → {SERVING_URL}')